# Chapter 9: Text Processing and Multiclass Classification

## 📋 Summary

Text is one of the most abundant and valuable forms of unstructured data. Before applying ML algorithms to text, it must be converted into numerical representations. This chapter covers the full pipeline: text cleaning and preprocessing, vectorization techniques (Bag of Words, TF-IDF, n-grams), feature extraction, text classification models, and multiclass classification strategies.

---

## 🧠 Theoretical Explanation

### Text Vectorization

**CountVectorizer (Bag of Words)**:
Creates a matrix where each row is a document and each column is a word. Cell values are raw word counts. Problems: ignores word order, common words dominate.

**TF-IDF (Term Frequency-Inverse Document Frequency)**:
Weights words by their importance:

`TF(t,d) = count(t in d) / total words in d`

`IDF(t) = log(N / df(t))` where N = total docs, df(t) = docs containing t

`TF-IDF(t,d) = TF(t,d) × IDF(t)`

Words common across all documents (like "the") get low IDF. Rare but meaningful words get high IDF.

**N-grams**: Instead of single words (unigrams), capture sequences of N words. Bigrams (N=2) capture phrases like "not good" vs just "good" and "not".

### Text Preprocessing
Typical steps:
1. Lowercasing
2. Remove punctuation and special characters
3. Remove stop words ("the", "is", "at")
4. Stemming ("running" → "run") or Lemmatization
5. Tokenization

### Multiclass Strategies
- **OvR (One-vs-Rest)**: One binary classifier per class
- **OvO (One-vs-One)**: One binary classifier per pair of classes
- **Multinomial models**: Directly support multiple classes (Naive Bayes, SGD, etc.)


## 9.1 Text Preprocessing

In [ ]:
import re
import string

# Sample texts
corpus = [
    "The quick brown fox jumps over the lazy dog!",
    "Machine learning is amazing and fun.",
    "scikit-learn makes ML easy for Python developers.",
    "Natural language processing is a core ML task.",
    "Text classification requires feature extraction."
]

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    text = re.sub(r'\d+', '', text)       # remove numbers
    text = text.strip()
    return text

cleaned = [preprocess_text(t) for t in corpus]
for original, clean in zip(corpus, cleaned):
    print(f'Original: {original}')
    print(f'Cleaned:  {clean}\n')

## 9.2 CountVectorizer (Bag of Words)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

vectorizer = CountVectorizer(stop_words='english')
X_counts = vectorizer.fit_transform(cleaned)

print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')
print(f'Matrix shape: {X_counts.shape}')

# Show as DataFrame
bow_df = pd.DataFrame(
    X_counts.toarray(),
    columns=vectorizer.get_feature_names_out()
)
bow_df

## 9.3 TF-IDF Vectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', max_features=20)
X_tfidf = tfidf.fit_transform(cleaned)

tfidf_df = pd.DataFrame(
    X_tfidf.toarray().round(4),
    columns=tfidf.get_feature_names_out()
)
print('TF-IDF Matrix:')
tfidf_df

## 9.4 Text Classification Pipeline (20 Newsgroups)

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

# Use 4 categories for a manageable multiclass problem
categories = ['alt.atheism', 'sci.space', 'talk.politics.guns', 'rec.sport.hockey']
train = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))
test = fetch_20newsgroups(subset='test', categories=categories, remove=('headers', 'footers', 'quotes'))

# Pipeline: TF-IDF + Naive Bayes
text_clf = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=10000, ngram_range=(1, 2))),
    ('clf', MultinomialNB())
])

text_clf.fit(train.data, train.target)
y_pred = text_clf.predict(test.data)

print(classification_report(test.target, y_pred, target_names=categories))

## 9.5 N-grams and Feature Extraction

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import SGDClassifier

# Compare unigrams vs bigrams
for ngram in [(1,1), (1,2), (2,2)]:
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=ngram, max_features=10000)),
        ('clf', SGDClassifier(random_state=42, max_iter=100))
    ])
    score = cross_val_score(pipe, train.data + test.data, 
                            list(train.target) + list(test.target), cv=3).mean()
    print(f'N-gram range {ngram}: {score:.4f}')

## 9.6 Evaluating Text Models

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test.target, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=categories, yticklabels=categories)
plt.title('Confusion Matrix - Text Classification')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

## 🔑 Key Takeaways

- **CountVectorizer** gives raw word counts; **TfidfVectorizer** gives importance-weighted counts.
- TF-IDF is almost always better than raw counts — it down-weights common words automatically.
- **N-grams** capture word sequences and often improve accuracy for phrase-sensitive tasks.
- **Multinomial Naive Bayes** is a fast, strong baseline for text classification.
- **SGDClassifier** with TF-IDF scales well to large text corpora.
- Always use a `Pipeline` — it ensures the vectorizer is fit only on training data.
